# 라우터 ELECTRA 분류기 — 파인튜닝 파이프라인 (Colab)

**⚠️ 실행 전 반드시 읽을 것 — 학습/평가 스키마가 서로 다릅니다.**

1. **학습셋 = `testset_natural_450.jsonl`** (in-scope 전용). 필드: `question`(텍스트), `business_function`(라벨, 실제 6업무). `is_oos`/`group` 없음(전부 in-scope).
2. **평가셋 = `testset_router_oos_1150.jsonl`** (MSP/AUROC용, 선택). 필드: `text`, `domain`(D1~D6), `is_oos`, `group`(G5a=far/G5b=near). **학습셋과 라벨 체계가 다릅니다**(업무명 vs D코드).
3. 이 노트북은 두 스키마를 **헬퍼(`get_text`/`get_label`)로 모두 처리**합니다. 그래서:
   - **near-OOD AUROC(G5b)** 는 라벨 매핑 없이 **바로 나옵니다** — is_oos와 MSP만 쓰므로.
   - **평가셋의 6-class 도메인 정확도**는 학습 라벨(업무명)과 평가 라벨(D코드)이 안 맞아 **자동 skip**됩니다. D코드↔업무 매핑(특히 D5)이 팀에서 정해지면 그때 활성화하세요.
4. 4개 후보는 전부 `AutoModelForSequenceClassification` 호환이라 `SELECTED_MODEL_KEY`만 바꿔 재실행하면 됩니다. `dialog-koelectra`는 small(~14M)만 공개라 나머지 base(110M)와 규모 차이 감안.

In [ ]:
# 1. 환경 설정 — 버전 고정(코랩 기본 transformers와 TrainingArguments 인자명이 달라 충돌하는 걸 방지)
!pip install -q -U transformers==4.44.2 accelerate==0.33.0 scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 92.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.1/315.1 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 148.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 123.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 127.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is

**⚠️ 위 pip install 셀 실행 직후, 바로 아래 셀을 실행해 런타임을 강제 재시작하세요.**

`ModuleNotFoundError: No module named 'numpy.rec'` 같은 오류는 코드 문제가 아니라, 같은 커널 안에서 numpy 계열 패키지를 pip install한 뒤 재시작 없이 바로 import해서 생기는 코랩 특유의 증상입니다(이미 로드된 numpy C-확장과 새로 깔린 numpy 바이너리가 어긋남). 아래 셀이 프로세스를 강제 종료시켜 코랩이 새 런타임으로 자동 재시작하게 만듭니다 — 재시작 후에는 pip install 셀을 다시 실행할 필요 없이 **다음 셀(임포트)부터 이어서 실행**하세요.

In [ ]:
import os
os.kill(os.getpid(), 9)  # 코랩이 자동으로 새 런타임을 띄운다. 이후 다음 셀부터 이어서 실행.

In [ ]:
import json
import random
from pathlib import Path

import numpy as np
import torch
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# 스키마 흡수 헬퍼 — 450(question/business_function)과 1150(text/domain) 모두 처리
def get_text(r: dict) -> str:
    return r.get("text") or r.get("question")


def get_label(r: dict):
    # 학습셋은 business_function, 평가셋(1150)은 domain
    return r.get("business_function") or r.get("domain")


def is_oos(r: dict) -> bool:
    return bool(r.get("is_oos", False))  # 450엔 필드 없음 → 전부 in-scope


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

device: cuda


In [ ]:
# 2. 설정 — 실험 전반에 걸쳐 고정되는 값만 여기 둔다.
#    LR/EPOCHS/WEIGHT_DECAY/MAX_LEN/dropout/label_smoothing 같은 튜닝 대상 하이퍼파라미터는
#    6번 셀 run_experiment()에서 실험마다 인자로 넘긴다 (가이드 §6 "단일 변경 → 관찰" 원칙을
#    코드로 옮긴 것 — 하이퍼파라미터를 여기 전역변수로 박아두면 바꿀 때마다 셀을 오가야 해서 비교가 안 됨).

MODEL_CANDIDATES = {
    "koelectra-base": "monologg/koelectra-base-v3-discriminator",
    "koelectra-small": "monologg/koelectra-small-v3-discriminator",
    "kcelectra-base": "beomi/KcELECTRA-base-v2022",
    "kr-electra": "snunlp/KR-ELECTRA-discriminator",
    "dialog-koelectra": "skplanet/dialog-koelectra-small-discriminator",  # small(~14M)만 공개
}

SELECTED_MODEL_KEY = "kcelectra-base"  # <- 4개 후보 전부 같은 코드로 실험 (여기만 변경)
MODEL_NAME = MODEL_CANDIDATES[SELECTED_MODEL_KEY]
assert MODEL_NAME is not None, f"'{SELECTED_MODEL_KEY}'의 모델 id가 비어 있습니다."

# 데이터 경로 — /content/data 에 아래 파일을 올린 뒤 실행
DATA_DIR = Path("/content/data")
TRAIN_PATH = DATA_DIR / "testset_natural_450.jsonl"       # 학습(in-scope, question+business_function) — 필수
VAL_PATH = DATA_DIR / "router_val.jsonl"                  # 없으면 TRAIN에서 자동 분할
TEST_PATH = DATA_DIR / "router_testset_v3_1.jsonl"    # MSP/AUROC 평가(1150, text+is_oos+group) — 선택

SEED = 42              # 다중시드 보고 시 42,1,7,100,2024 등으로 반복
VAL_RATIO = 0.2
OUTPUT_DIR = Path("/content/router_electra_out")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

set_seed(SEED)
print("selected model:", MODEL_NAME)

In [ ]:
# 3. 데이터 로딩 (스키마는 헬퍼로 흡수 — 450/1150 모두 가능)


def load_jsonl(path: Path) -> list:
    if not path.exists():
        return []
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


train_rows = load_jsonl(TRAIN_PATH)
assert train_rows, (
    f"{TRAIN_PATH}가 비어있거나 없습니다. testset_natural_450.jsonl 을 이 경로에 올린 뒤 다시 실행하세요."
)

val_rows = load_jsonl(VAL_PATH)
if not val_rows:
    random.Random(SEED).shuffle(train_rows)
    n_val = max(1, int(len(train_rows) * VAL_RATIO))
    val_rows, train_rows = train_rows[:n_val], train_rows[n_val:]

test_rows = load_jsonl(TEST_PATH)  # 없으면 8번 셀(MSP/AUROC)은 건너뜀

# G6 제외 — 라벨링 자체가 애매한 저신뢰 OOS 그룹이라 채점(AUROC)에 섞이면 결과가 왜곡됨.
# oos로 라벨링되어 있어 근처 in-scope와 구분이 어려운 문항들 → 채점 대상에서 제거하고 학습에는 애초에 안 씀.
_n_before_g6 = len(test_rows)
test_rows = [r for r in test_rows if r.get("group") != "G6"]
_n_g6 = _n_before_g6 - len(test_rows)
if _n_g6:
    print(f"G6(저신뢰 라벨) {_n_g6}건을 테스트셋/채점 대상에서 제외했습니다.")

print(f"train={len(train_rows)}  val={len(val_rows)}  test(mixed)={len(test_rows)}")

In [ ]:
# 4. 라벨 체계 — 학습 데이터의 라벨(business_function/domain)에서 자동 도출

LABELS = sorted({get_label(r) for r in train_rows if get_label(r) and not is_oos(r)})
assert LABELS, "학습 데이터에 in-scope 라벨(business_function 또는 domain)이 없습니다."
label2id = {lab: i for i, lab in enumerate(LABELS)}
id2label = {i: lab for lab, i in label2id.items()}
print("LABELS:", LABELS)

train_rows = [r for r in train_rows if get_label(r) and not is_oos(r)]
val_rows = [r for r in val_rows if get_label(r) and not is_oos(r)]
print(f"학습에 실제 사용되는 in-scope 건수: train={len(train_rows)}  val={len(val_rows)}")

LABELS: ['고객 미수령금 신청', '예금보험금 안내', '예금자보호제도', '은닉재산 신고', '착오송금 반환 신청', '채무조정 안내']
학습에 실제 사용되는 in-scope 건수: train=360  val=90


In [ ]:
# 5. 토크나이저 — 모델 하나로 고정이라 한 번만 로드해서 모든 실험에서 재사용
#    (max_len은 실험마다 달라질 수 있어 Dataset 생성 시점에 인자로 받는다)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class RouterDataset(Dataset):
    def __init__(self, rows: list, max_len: int):
        texts = [get_text(r) for r in rows]
        self.labels = [label2id[get_label(r)] for r in rows]
        enc = tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            max_length=max_len,
        )
        self.input_ids = enc["input_ids"]
        self.attention_mask = enc["attention_mask"]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": torch.tensor(self.input_ids[idx]),
            "attention_mask": torch.tensor(self.attention_mask[idx]),
            "labels": torch.tensor(self.labels[idx]),
        }


print(f"train={len(train_rows)}  val={len(val_rows)}  (토큰화는 실험마다 run_experiment 안에서 수행)")

In [ ]:
# 6. 실험 러너 — 모델 로딩+학습+평가를 하나로 묶어서, 하이퍼파라미터 조합 하나를 통째로 실행하고
#    (val accuracy/macro_f1, G5a/G5b AUROC)를 반환한다. 가이드 §6 "단일 변경 → 관찰"을 그대로 코드로 옮긴 것.
#    metric_for_best_model은 가이드 §3 지시대로 macro_f1 고정(val엔 OOS가 없어 AUROC로 못 바꿈).


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }


def build_training_args(**kw):
    try:
        return TrainingArguments(eval_strategy="epoch", **kw)
    except TypeError:
        return TrainingArguments(evaluation_strategy="epoch", **kw)


def _fmt(x):
    return f"{x:.3f}" if x is not None else "n/a"


def _group_auroc(msp, rows, group, oos):
    neg = ~oos
    pos = np.array([o and r.get("group") == group for o, r in zip(oos, rows)])
    if pos.sum() < 1 or neg.sum() < 1:
        return None
    mask = pos | neg
    return roc_auc_score(oos[mask], (1 - msp)[mask])


def run_experiment(
    name: str,
    lr: float = 2e-5,
    epochs: int = 8,
    batch_size: int = 16,
    weight_decay: float = 0.01,
    max_len: int = 64,
    warmup_ratio: float = 0.0,
    label_smoothing: float = 0.0,
    hidden_dropout_prob=None,
    classifier_dropout=None,
    use_class_weights: bool = True,
    seed: int = SEED,
) -> dict:
    """하이퍼파라미터 조합 하나를 처음부터 학습·평가하고 결과 dict를 반환."""
    print(f"\n{'='*78}\n[실험: {name}]  lr={lr} epochs={epochs} bs={batch_size} wd={weight_decay} "
          f"max_len={max_len} warmup={warmup_ratio} label_smoothing={label_smoothing} "
          f"hidden_dropout={hidden_dropout_prob} classifier_dropout={classifier_dropout}\n{'='*78}")

    set_seed(seed)
    tr_ds = RouterDataset(train_rows, max_len)
    va_ds = RouterDataset(val_rows, max_len)

    model_kwargs = dict(num_labels=len(LABELS), id2label=id2label, label2id=label2id)
    if hidden_dropout_prob is not None:
        model_kwargs["hidden_dropout_prob"] = hidden_dropout_prob
    if classifier_dropout is not None:
        model_kwargs["classifier_dropout"] = classifier_dropout
    m = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, **model_kwargs).to(DEVICE)
    for _p in m.parameters():
        _p.data = _p.data.contiguous()
    print("dropout 적용확인:", m.config.hidden_dropout_prob, m.config.classifier_dropout)

    class_weights = None
    if use_class_weights:
        counts = np.bincount([label2id[get_label(r)] for r in train_rows], minlength=len(LABELS))
        class_weights = torch.tensor(counts.sum() / (len(LABELS) * counts), dtype=torch.float32).to(DEVICE)

    class RouterTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            labels = inputs.pop("labels")
            outputs = model(**inputs)
            loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights, label_smoothing=label_smoothing)
            loss = loss_fct(outputs.logits.view(-1, len(LABELS)), labels.view(-1))
            return (loss, outputs) if return_outputs else loss

    args = build_training_args(
        output_dir=str(OUTPUT_DIR / "checkpoints" / name),
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        learning_rate=lr,
        weight_decay=weight_decay,
        warmup_ratio=warmup_ratio,
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        logging_steps=50,
        seed=seed,
        report_to="none",
        save_safetensors=False,  # ELECTRA + safetensors 조합에서 나는 "non contiguous tensor" 오류 회피
        disable_tqdm=True,
    )
    trainer = RouterTrainer(
        model=m, args=args, train_dataset=tr_ds, eval_dataset=va_ds,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )
    trainer.train()
    val_metrics = trainer.evaluate()

    # near-OOD 평가 — G6은 3번 셀에서 이미 제외된 test_rows 사용
    m.eval()
    with torch.no_grad():
        texts = [get_text(r) for r in test_rows]
        enc = tokenizer(texts, truncation=True, padding=True, max_length=max_len, return_tensors="pt").to(DEVICE)
        msp = torch.softmax(m(**enc).logits, dim=-1).cpu().numpy().max(axis=-1)
    oos = np.array([is_oos(r) for r in test_rows])
    g5b = _group_auroc(msp, test_rows, "G5b", oos)
    g5a = _group_auroc(msp, test_rows, "G5a", oos)

    result = {
        "name": name,
        "val_accuracy": val_metrics.get("eval_accuracy"),
        "val_macro_f1": val_metrics.get("eval_macro_f1"),
        "g5a_auroc": g5a,
        "g5b_auroc": g5b,
        "model": m,
        "max_len": max_len,
    }
    print(f"[결과: {name}] val_acc={_fmt(result['val_accuracy'])}  val_macro_f1={_fmt(result['val_macro_f1'])}  "
          f"G5a={_fmt(g5a)}  G5b={_fmt(g5b)}  <- near-OOD 승부처")
    return result

In [ ]:
# 7. 가이드 §4 우선순위대로 baseline 위에 한 축씩 누적 적용 — 단일 변경마다 G5b AUROC를 직접 비교
#    baseline = 가이드 §1 표의 "현재" 값 그대로 (LR=2e-5, EPOCHS=8, BATCH_SIZE=16, WEIGHT_DECAY=0.01, MAX_LEN=64)
#    BATCH_SIZE/MAX_LEN은 가이드가 "적용대상: 정확도"로 분류해 §4 우선순위에 안 넣은 축이라 전 실험에서 고정.

results = []

results.append(run_experiment(
    "0_baseline",
    lr=2e-5, epochs=8, batch_size=16, weight_decay=0.01, max_len=64,
))

results.append(run_experiment(
    "1_label_smoothing",  # §4-1 최우선: 정답에도 100% 확신 대신 약하게 → near 분리력에 직접 효과
    lr=2e-5, epochs=8, batch_size=16, weight_decay=0.01, max_len=64,
    label_smoothing=0.1,
))

results.append(run_experiment(
    "2_epochs_weightdecay",  # §4-2: 위 + 과적합 최소화
    lr=2e-5, epochs=5, batch_size=16, weight_decay=0.05, max_len=64,
    label_smoothing=0.1,
))

results.append(run_experiment(
    "3_dropout",  # §4-3: 위 + 분류 헤드 과신 억제
    lr=2e-5, epochs=5, batch_size=16, weight_decay=0.05, max_len=64,
    label_smoothing=0.1, hidden_dropout_prob=0.2, classifier_dropout=0.3,
))

results.append(run_experiment(
    "4_full_recipe_section5",  # §5 코드 반영 요약: LR/MAX_LEN/warmup까지 가이드 번들 그대로
    lr=1e-5, epochs=5, batch_size=16, weight_decay=0.05, max_len=96,
    warmup_ratio=0.1, label_smoothing=0.1, hidden_dropout_prob=0.2, classifier_dropout=0.3,
))

print(f"\n{'='*78}\n실험 결과 비교 (near-OOD G5b AUROC 기준 내림차순)\n{'='*78}")
for r in sorted(results, key=lambda x: (x["g5b_auroc"] is not None, x["g5b_auroc"] or -1), reverse=True):
    print(f"{r['name']:<28} val_acc={_fmt(r['val_accuracy'])}  val_f1={_fmt(r['val_macro_f1'])}  "
          f"G5a={_fmt(r['g5a_auroc'])}  G5b={_fmt(r['g5b_auroc'])}")

best = max(results, key=lambda x: x["g5b_auroc"] if x["g5b_auroc"] is not None else -1)
model = best["model"]
MAX_LEN = best["max_len"]  # 8/9번 셀의 predict_msp가 이 MAX_LEN을 그대로 사용
print(f"\n>>> 최고 G5b AUROC: {best['name']}  (G5b={_fmt(best['g5b_auroc'])}) — model을 이 실험으로 교체했습니다.")
print("    ⚠️ 단일 시드(42) 결과. 가이드 §6대로 최종 채택 전엔 시드 3~5개 평균±표준편차로 재검증 권장.")

In [ ]:
# 8. MSP 기반 OOS 평가 — near-OOD(G5b)가 핵심 지표
#    도메인 정확도는 학습라벨(업무명)과 평가라벨(D코드)이 같을 때만 계산(아니면 skip)


@torch.no_grad()
def predict_msp(rows: list):
    """각 문항에 대해 (argmax index, MSP=max softmax prob) 반환."""
    model.eval()
    texts = [get_text(r) for r in rows]
    enc = tokenizer(texts, truncation=True, padding=True, max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
    logits = model(**enc).logits
    probs = torch.softmax(logits, dim=-1).cpu().numpy()
    return probs.argmax(axis=-1), probs.max(axis=-1)


def evaluate_router(test_rows: list) -> None:
    if not test_rows:
        print("테스트셋이 비어 있어 MSP/AUROC 평가를 건너뜁니다.")
        return

    pred_idx, msp = predict_msp(test_rows)
    oos = np.array([is_oos(r) for r in test_rows])

    # (a) in-scope 6-class 정확도 — 평가 라벨이 학습 LABELS와 같을 때만
    in_scope_rows = [(i, r) for i, r in enumerate(test_rows)
                     if not is_oos(r) and get_label(r) in label2id]
    if in_scope_rows:
        idxs, rows_ = zip(*in_scope_rows)
        y_true = [label2id[get_label(r)] for r in rows_]
        y_pred = [pred_idx[i] for i in idxs]
        print(f"[in-scope] n={len(y_true)}  accuracy={accuracy_score(y_true, y_pred):.3f}  "
              f"macro_f1={f1_score(y_true, y_pred, average='macro'):.3f}")
        print(confusion_matrix(y_true, y_pred))
    else:
        print("[in-scope] 도메인 정확도 skip — 학습 라벨(업무명)과 평가 라벨(D코드) 체계가 다름. "
              "D코드↔업무 매핑 확정 후 활성화.")

    # (b) OOS 탐지 — MSP 낮을수록 OOS이므로 score=1-msp (라벨 매핑 불필요)
    if oos.any() and (~oos).any():
        score = 1 - msp
        print(f"[OOS 탐지] 전체 AUROC={roc_auc_score(oos, score):.3f}")

        # in-scope(음성) vs 각 OOS 그룹(양성) 분리 — far(G5a)/near(G5b)
        neg = ~oos
        for g in sorted({r.get("group") for r in test_rows if is_oos(r)}):
            pos = np.array([is_oos(r) and r.get("group") == g for r in test_rows])
            m = pos | neg
            if pos.sum() >= 1 and neg.sum() >= 1:
                tag = " <- near-OOD 승부처" if g == "G5b" else (" (far-OOD)" if g == "G5a" else "")
                print(f"   {g} AUROC={roc_auc_score(oos[m], score[m]):.3f}  (n_oos={int(pos.sum())}){tag}")
    else:
        print("테스트셋에 in-scope/OOS가 둘 다 있어야 AUROC를 계산할 수 있습니다.")


evaluate_router(test_rows)

[in-scope] 도메인 정확도 skip — 학습 라벨(업무명)과 평가 라벨(D코드) 체계가 다름. D코드↔업무 매핑 확정 후 활성화.
[OOS 탐지] 전체 AUROC=0.557
   G5a AUROC=0.787  (n_oos=80) (far-OOD)
   G5b AUROC=0.598  (n_oos=260) <- near-OOD 승부처
   G6 AUROC=0.340  (n_oos=133)


In [ ]:
# 9. 모델 저장 (+ 코랩 밖으로 내보내기용 zip)

final_dir = OUTPUT_DIR / "final"
model.save_pretrained(final_dir, safe_serialization=False)
tokenizer.save_pretrained(final_dir)

import shutil
zip_path = shutil.make_archive(str(OUTPUT_DIR / "router_electra_model"), "zip", final_dir)
print("saved:", zip_path)

# 구글 드라이브에 보관하려면:
# from google.colab import drive
# drive.mount('/content/drive')
# shutil.copy(zip_path, '/content/drive/MyDrive/')

saved: /content/router_electra_out/router_electra_model.zip


## 다음에 할 일

- **4개 후보 비교**: `SELECTED_MODEL_KEY`만 `koelectra-base` → `koelectra-small` → `kcelectra-base` → `kr-electra` → `dialog-koelectra`로 바꿔가며 이 노트북을 그대로 재실행하면 됩니다. `dialog-koelectra`는 small(~14M)만 있어 나머지 base(110M) 후보와 규모 차이가 있다는 점을 감안해서 비교하세요.
- **가이드 우선순위 실험(7번 셀)**: `run_experiment(name, **hparams)`를 호출하는 방식이라, 새 조합을 시험하려면 7번 셀에 `results.append(run_experiment("내이름", lr=..., epochs=..., ...))` 한 줄만 추가하면 됩니다. 매 실험마다 val accuracy/macro_f1과 G5a/G5b AUROC가 바로 출력되고, 실행이 끝나면 G5b 기준으로 정렬된 비교표가 나옵니다.
- **다중 시드 평균 (가이드 §6)**: 지금은 단일 시드(42) 결과입니다. 최종 채택 전엔 `run_experiment(..., seed=1)`, `seed=7`처럼 시드를 바꿔 같은 조합을 3~5회 반복하고 G5b AUROC를 평균±표준편차로 보고하세요. 단일 시드 최고값만으로 결론 내리지 말 것.
- **실행 전**: `DATA_DIR`(`/content/data`)에 `testset_natural_450.jsonl`(필수)·`router_val.jsonl`(선택)·`router_testset_v3_1.jsonl`(선택, MSP/AUROC용)을 올려두면 나머지 셀은 수정 없이 그대로 돌아갑니다.